In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [21]:
# Loading accident data 
accident_list = []
for year in range(2019, 2025):
    file = f"FARS{year}NationalCSV/accident.csv"
    df = pd.read_csv(file, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.replace("ï»¿", "", regex=False)
    df["YEAR"] = year
    accident_list.append(df)

#Combining all years into one dataset
accident = pd.concat(accident_list, ignore_index=True)
print("Total crashes:", len(accident))
print(accident.groupby("YEAR").size())

Total crashes: 222695
YEAR
2019    33487
2020    35935
2021    39785
2022    39422
2023    37769
2024    36297
dtype: int64


In [22]:
#Checking for duplicate crashes
duplicates = accident.duplicated(subset=["YEAR", "ST_CASE"]).sum()
print("Duplicate crashes:", duplicates)
print("Total crashes:", len(accident))

Duplicate crashes: 0
Total crashes: 222695


In [31]:
#Selecting variables that are relevant to our project
columns = ["YEAR", "ST_CASE", "MONTH", "DAY", "HOUR", "DAY_WEEK", "STATE", "WEATHER"]
crashes = accident[columns].copy()

# Checking missing values
print(crashes.isnull().sum())

YEAR        0
ST_CASE     0
MONTH       0
DAY         0
HOUR        0
DAY_WEEK    0
STATE       0
WEATHER     0
dtype: int64


In [32]:
# Checking the values 
print("Hour:")
print(crashes["HOUR"].value_counts().sort_index())

print("\nWeather:")
print(crashes["WEATHER"].value_counts().sort_index())

Hour:
HOUR
0      9073
1      8088
2      8298
3      6201
4      5676
5      7100
6      7960
7      7047
8      5857
9      5901
10     6663
11     7320
12     8448
13     8896
14     9987
15    10957
16    11042
17    12084
18    13188
19    12764
20    13307
21    13272
22    11642
23    10256
99     1668
Name: count, dtype: int64

Weather:
WEATHER
1     161635
2      15395
3        220
4       1809
5       2349
6        357
7         59
8        194
10     29077
11       183
12       183
98     10160
99      1074
Name: count, dtype: int64


In [33]:
# Cleaning crash hour
crashes["HOUR"] = pd.to_numeric(crashes["HOUR"], errors="coerce")
crashes.loc[~crashes["HOUR"].between(0, 23), "HOUR"] = np.nan

# Checking our result
print("Missing or unknown hours:", crashes["HOUR"].isnull().sum())

# Checking weather codes for each year
print(pd.crosstab(crashes["WEATHER"], crashes["YEAR"]))

Missing or unknown hours: 1668
YEAR      2019   2020   2021   2022   2023   2024
WEATHER                                          
1        22666  25189  28977  29619  28023  27161
2         2592   2642   2616   2427   2663   2455
3           47     27     33     40     45     28
4          331    283    305    374    222    294
5          350    372    476    399    399    353
6           51     56     68     71     45     66
7            9      6     15      9      9     11
8           30     20     46     45     35     18
10        4857   4648   5413   4814   4876   4469
11          25     26     28     47     36     21
12          37     38     29     40     17     22
98        2325   2430   1604   1296   1263   1242
99         167    198    175    241    136    157


In [34]:
# Cleaning weather conditions
crashes["WEATHER"] = crashes["WEATHER"].replace([98, 99], np.nan)

print("Missing or unknown weather:", crashes["WEATHER"].isnull().sum())

Missing or unknown weather: 11234
